In [ ]:
%%capture
!pip install pymupdf pandas python-dotenv openai tqdm
!pip install -U google-api-python-client google-auth-httplib2 google-auth-oauthlib
!pip install easyocr pdf2image pypdf
!pip install gdown
!apt-get install -y poppler-utils

In [ ]:
import gdown
import os

# 1. Google Drive link to download invoices
drive_url = "Your Drive URL"

# 2. Define the local folder name
output_folder = "invoices"

# 3. Create the folder if it doesn't already exist
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    print(f"Created directory: {output_folder}")

# 4. Download the contents into the 'invoices'
gdown.download_folder(drive_url, output=output_folder, quiet=False, use_cookies=False)

Created directory: invoices


Retrieving folder contents


Processing file 1L6i3ZD3ARcmm8LjhFl_1qqksSeL3ifng Copy of ARPFIINVOEBTCHLASER (1).pdf
Processing file 1-ZnLgZ1nBrgIgT9oFyNdykquEgZZLukR Copy of ARPFIINVOEBTCHLASER (2).pdf
Processing file 1ZmRdVXqvb-tse4pc8MUKfTla5WZnAcCL Copy of ARPFIINVOEBTCHLASER (3).pdf
Processing file 14Jp8599xbUqOCGbKnVQPOigTMm0L1TBR Copy of ARPFIINVOEBTCHLASER (4).pdf
Processing file 1TwNhSTt6HjtWkr07tLWTWbvL-1QQm8eY Copy of ARPFIINVOEBTCHLASER (5).pdf
Processing file 1BntRm9_iLlUBRMXcNkWgfenIi0ZwkuTa Copy of ARPFIINVOEBTCHLASER (6).pdf
Processing file 1XN_4bokyPtuXCOKua9chg6IDuk7uqnQx Copy of ARPFIINVOEBTCHLASER (7).pdf
Processing file 1-GduiZQVYwTvCvce67bqts2YEzBwb-FH Copy of ARPFIINVOEBTCHLASER (8).pdf
Processing file 1inbal8MhzQ3e2qgUQvaIHJjvPLdbMoum Copy of ARPFIINVOEBTCHLASER (9).pdf
Processing file 1GC21wGevl169oMZG5E3H8KRTQ0SP39o5 Copy of ARPFIINVOEBTCHLASER (10).pdf
Processing file 1ZxgDDwtf_JwGRdTUEgY5A5gI6v-eq44_ Copy of ARPFIINVOEBTCHLASER (11).pdf
Processing file 1Z1MhINKP0XKPsfeMiQTWclXBnPjYCgOS Co

Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1L6i3ZD3ARcmm8LjhFl_1qqksSeL3ifng
To: /content/invoices/Copy of ARPFIINVOEBTCHLASER (1).pdf
100%|██████████| 39.4k/39.4k [00:00<00:00, 68.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1-ZnLgZ1nBrgIgT9oFyNdykquEgZZLukR
To: /content/invoices/Copy of ARPFIINVOEBTCHLASER (2).pdf
100%|██████████| 39.3k/39.3k [00:00<00:00, 65.5MB/s]
Downloading...
From: https://drive.google.com/uc?id=1ZmRdVXqvb-tse4pc8MUKfTla5WZnAcCL
To: /content/invoices/Copy of ARPFIINVOEBTCHLASER (3).pdf
100%|██████████| 39.3k/39.3k [00:00<00:00, 49.2MB/s]
Downloading...
From: https://drive.google.com/uc?id=14Jp8599xbUqOCGbKnVQPOigTMm0L1TBR
To: /content/invoices/Copy of ARPFIINVOEBTCHLASER (4).pdf
100%|██████████| 46.5k/46.5k [00:00<00:00, 72.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1TwNhSTt6HjtWkr07tLWTWbvL-1QQm8eY
To: /content/invoices

['invoices/Copy of ARPFIINVOEBTCHLASER (1).pdf',
 'invoices/Copy of ARPFIINVOEBTCHLASER (2).pdf',
 'invoices/Copy of ARPFIINVOEBTCHLASER (3).pdf',
 'invoices/Copy of ARPFIINVOEBTCHLASER (4).pdf',
 'invoices/Copy of ARPFIINVOEBTCHLASER (5).pdf',
 'invoices/Copy of ARPFIINVOEBTCHLASER (6).pdf',
 'invoices/Copy of ARPFIINVOEBTCHLASER (7).pdf',
 'invoices/Copy of ARPFIINVOEBTCHLASER (8).pdf',
 'invoices/Copy of ARPFIINVOEBTCHLASER (9).pdf',
 'invoices/Copy of ARPFIINVOEBTCHLASER (10).pdf',
 'invoices/Copy of ARPFIINVOEBTCHLASER (11).pdf',
 'invoices/Copy of ARPFIINVOEBTCHLASER (12).pdf',
 'invoices/Copy of ARPFIINVOEBTCHLASER (13).pdf',
 'invoices/Copy of ARPFIINVOEBTCHLASER (14).pdf',
 'invoices/Copy of ARPFIINVOEBTCHLASER.pdf',
 'invoices/Copy of Inv_20065629_from_Franks_Quality_Produce_18417572_22492.pdf',
 'invoices/Copy of Inv_20066901_from_Franks_Quality_Produce_18471648_18416.pdf',
 'invoices/Copy of Inv_20068140_from_Franks_Quality_Produce_18529019_11828.pdf',
 'invoices/Copy of In

In [ ]:
import easyocr
import os
import json
import openai
import pypdf
from PIL import Image
from pdf2image import convert_from_path
from google.colab import userdata
from pydantic import BaseModel
from typing import List

In [ ]:
# 2. Define Data Schema
class LineItem(BaseModel):
    description: str
    quantity: float
    unit_price: float
    line_total: float

class InvoiceData(BaseModel):
    invoice_number: str
    date: str
    vendor: str
    total_amount: float
    line_items: List[LineItem]

# 3. Setup OpenRouter Client
api_key = userdata.get('OPENROUTER_API_KEY')
client = openai.OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)

reader = easyocr.Reader(['en'])

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

In [ ]:
import fitz  # PyMuPDF
import os

input_folder = "/content/invoices"
output_folder = "Invoices_images"

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

for filename in os.listdir(input_folder):
    if filename.lower().endswith(".pdf"):
        pdf_path = os.path.join(input_folder, filename)

        # Open the PDF
        doc = fitz.open(pdf_path)

        for i, page in enumerate(doc):
            # Increase resolution (2.0 = 2x zoom/DPI)
            # This makes the OCR much more accurate
            zoom = 2.0
            mat = fitz.Matrix(zoom, zoom)
            pix = page.get_pixmap(matrix=mat)

            # Define image name
            base_name = os.path.splitext(filename)[0]
            image_path = os.path.join(output_folder, f"{base_name}_pg{i+1}.png")

            # Save the image
            pix.save(image_path)

        doc.close()
        print(f"Converted: {filename}")

print("\n✅ All PDFs converted to images in the /invoice_images folder.")

Converted: Copy of ARPFIINVOEBTCHLASER (1).pdf
Converted: Copy of Inv_20075310_from_Franks_Quality_Produce_18842235_13952 (1).pdf
Converted: Copy of ARPFIINVOEBTCHLASER (14).pdf
Converted: Copy of Inv_20074433_from_Franks_Quality_Produce_18802791_20408 (1).pdf
Converted: Copy of ARPFIINVOEBTCHLASER (4).pdf
Converted: Copy of Inv_20068978_from_Franks_Quality_Produce_18559799_11828.pdf
Converted: Copy of Inv_20075310_from_Franks_Quality_Produce_18842235_13952.pdf
Converted: Copy of ARPFIINVOEBTCHLASER (9).pdf
Converted: Copy of Inv_20074433_from_Franks_Quality_Produce_18802791_20408.pdf
Converted: Copy of Inv_20077622_from_Franks_Quality_Produce_18946409_7596 (1).pdf
Converted: Copy of Inv_20068140_from_Franks_Quality_Produce_18529019_11828.pdf
Converted: Copy of Inv_20076301_from_Franks_Quality_Produce_18887473_13748 (1).pdf
Converted: Copy of ARPFIINVOEBTCHLASER (5).pdf
Converted: Copy of ARPFIINVOEBTCHLASER (10).pdf
Converted: Copy of Inv_20077622_from_Franks_Quality_Produce_18946409_

In [ ]:
import os
import json
import gc
import easyocr
import numpy as np

from PIL import Image
from tqdm import tqdm
from typing import List
from pydantic import BaseModel
from openai import OpenAI
from google.colab import userdata

In [ ]:
api_key = userdata.get("OPENROUTER_API_KEY")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)

In [ ]:
class LineItem(BaseModel):
    description: str
    quantity: float
    unit_price: float
    line_total: float

class InvoiceData(BaseModel):
    invoice_number: str
    date: str
    vendor: str
    total_amount: float
    line_items: List[LineItem]

In [ ]:
reader = easyocr.Reader(['en'], gpu=True)

In [ ]:
def extract_text_from_image(image_path):

    img = Image.open(image_path).convert("RGB")

    img.thumbnail((1500, 1500))   # 🔥 prevents RAM crash

    img_np = np.array(img)

    results = reader.readtext(img_np, detail=0, paragraph=True)

    del img, img_np
    gc.collect()

    return "\n".join(results)

In [ ]:
def process_invoice_ai(text):

    text = text[:6000]   # 🔥 reduce token cost & memory

    prompt = f"""
Extract invoice data into STRICT JSON.

Return ONLY valid JSON.

Schema:
{InvoiceData.model_json_schema()}

TEXT:
{text}
"""

    response = client.chat.completions.create(
        model="stepfun/step-3.5-flash:free",
        messages=[{"role": "user", "content": prompt}]
    )

    return json.loads(response.choices[0].message.content)

In [ ]:
!unzip invoices_images.zip

unzip:  cannot find or open invoices_images.zip, invoices_images.zip.zip or invoices_images.zip.ZIP.


In [ ]:
import os

print("Folder exists:", os.path.exists(input_folder))
print("Files:", os.listdir(input_folder))

Folder exists: True
Files: ['Copy of ARPFIINVOEBTCHLASER (2)_pg1.png', 'Copy of ARPFIINVOEBTCHLASER (10)_pg4.png', 'Copy of ARPFIINVOEBTCHLASER (11)_pg1.png', 'Copy of Inv_20078834_from_Franks_Quality_Produce_18994615_32132_pg1.png', 'Copy of ARPFIINVOEBTCHLASER (3)_pg1.png', 'Copy of Inv_20077622_from_Franks_Quality_Produce_18946409_7596_pg1.png', 'Copy of ARPFIINVOEBTCHLASER (7)_pg1.png', 'Copy of ARPFIINVOEBTCHLASER (10)_pg1.png', 'Copy of ARPFIINVOEBTCHLASER (8)_pg1.png', 'Copy of ARPFIINVOEBTCHLASER (11)_pg4.png', 'Copy of ARPFIINVOEBTCHLASER (4)_pg1.png', 'Copy of Inv_20076301_from_Franks_Quality_Produce_18887473_13748 (1)_pg1.png', 'Copy of Inv_20066901_from_Franks_Quality_Produce_18471648_18416_pg1.png', 'Copy of Inv_20075310_from_Franks_Quality_Produce_18842235_13952_pg1.png', 'Copy of ARPFIINVOEBTCHLASER (12)_pg1.png', 'Copy of ARPFIINVOEBTCHLASER (9)_pg3.png', 'Copy of ARPFIINVOEBTCHLASER (14)_pg1.png', 'Copy of ARPFIINVOEBTCHLASER (9)_pg2.png', 'Copy of ARPFIINVOEBTCHLASER_

In [ ]:
input_folder = "/content/Invoices_images"
output_folder = "/content/output_json"

os.makedirs(output_folder, exist_ok=True)

In [ ]:
all_results = []

image_files = [
    f for f in os.listdir(input_folder)
    if f.lower().endswith((".png", ".jpg", ".jpeg", ".webp"))
]

for file in tqdm(image_files):

    file_path = os.path.join(input_folder, file)

    print(f"\nProcessing → {file}")

    try:
        raw_text = extract_text_from_image(file_path)

        json_data = process_invoice_ai(raw_text)

        validated = InvoiceData(**json_data)

        all_results.append(validated.dict())

        with open(f"{output_folder}/{file}.json", "w") as f:
            json.dump(validated.dict(), f, indent=2)

    except Exception as e:
        print(f"❌ Failed → {file} → {e}")

    gc.collect()

  0%|          | 0/48 [00:00<?, ?it/s]


Processing → Copy of ARPFIINVOEBTCHLASER (2)_pg1.png


/tmp/ipython-input-232/1327713.py:21: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  all_results.append(validated.dict())
/tmp/ipython-input-232/1327713.py:24: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  json.dump(validated.dict(), f, indent=2)
  2%|▏         | 1/48 [02:45<2:09:33, 165.40s/it]


Processing → Copy of ARPFIINVOEBTCHLASER (10)_pg4.png


  4%|▍         | 2/48 [03:28<1:11:27, 93.21s/it] 

...

 96%|█████████▌| 46/48 [1:16:55<02:41, 80.53s/it]


Processing → Copy of Inv_20068978_from_Franks_Quality_Produce_18559799_11828_pg1.png


 98%|█████████▊| 47/48 [1:18:39<01:27, 87.77s/it]


Processing → Copy of Inv_20071084_from_Franks_Quality_Produce_18665028_7884_pg1.png


100%|██████████| 48/48 [1:19:53<00:00, 99.87s/it]


In [ ]:
with open("/content/all_invoices.json", "w") as f:
    json.dump(all_results, f, indent=2)

In [ ]:
all_results

[{'invoice_number': '378793',
  'date': '07/29/2025',
  'vendor': 'Pacific Food Importers Inc.',
  'total_amount': 754.89,
  'line_items': [{'description': 'FLOUR POWER GRAINCRAFT 50 LB',
    'quantity': 10.0001,
    'unit_price': 24.063,
    'line_total': 240.0},
   {'description': 'ONION MINCED MARCA CROC 4 * 4 .5 LB',
    'quantity': 1.0,
    'unit_price': 113.333,
    'line_total': 113.33},
   {'description': 'SESAME SEEDS MHITE+ MARCA CROC 4 LB',
    'quantity': 1.0,
    'unit_price': 80.25,
    'line_total': 80.25},
   {'description': 'SALT KOSHER COARSE DIAMOND CR 3 LB',
    'quantity': 2.0,
    'unit_price': 75.076,
    'line_total': 150.15},
   {'description': 'GLOVE NITRILE XL 10 100 ct',
    'quantity': 1.0,
    'unit_price': 36.25,
    'line_total': 36.25},
   {'description': 'YEAST INSTANT RED LBL SAF 20',
    'quantity': 1.0,
    'unit_price': 130.0,
    'line_total': 130.0}]},
 {'invoice_number': '376172',
  'date': '06/05/2025',
  'vendor': 'Pacific Flood mporters INc',

In [ ]:
import pandas as pd

In [ ]:
df=pd.read_json("/content/all_invoices.json")
df

,invoice_number,date,vendor,total_amount,line_items
0,378793,2025-07-29,Pacific Food Importers Inc.,754.89,[{'description': 'FLOUR POWER GRAINCRAFT 50 LB...
1,376172,2025-06-05,Pacific Flood mporters INc,133.42,[{'description': 'YEAST INSTANT RED LBL SAF 20...
2,375991,2025-06-03,Pacific Flood Importers Inc,491.83,[{'description': 'FLOUR POWER 50 LB GRAINCRAFT...
3,20078834,2025-08-14,Frank's QQuality Produce,53.35,"[{'description': 'ONION, GREEN', 'quantity': 2..."
4,378483,2025-07-22,Pacific Flood mporters INc,576.63,"[{'description': 'FLOUR POWER GRAINCRAFT LB', ..."
5,20077622,2025-08-06,Frank's QQuality Produce,79.11,"[{'description': 'TOMATO, ROMA', 'quantity': 1..."
6,376739,2025-06-17,Pacific Food Importers Inc.,385.06,[{'description': 'FLOUR POWER GRAINCRAFT 50 LB...
7,375991,2025-06-03,Pacific Food Importers Inc.,491.83,[{'description': 'FLOUR POWER 50 LB GRAINCRAFT...
8,376355,2025-06-10,Pacific Food Importers Inc,688.51,"[{'description': 'FLOUR POWER GRAINCRAFT', 'qu..."
9,376172,2025-06-05,Pacific Food Importers Inc.,133.42,[{'description': 'YEAST INSTANT RED LBL SAF 20...


In [ ]:
df.to_csv("invioce.csv")

In [ ]:
s=pd.read_csv("/content/invioce.csv")
s

,Unnamed: 0,invoice_number,date,vendor,total_amount,line_items
0,0,378793,2025-07-29,Pacific Food Importers Inc.,754.89,[{'description': 'FLOUR POWER GRAINCRAFT 50 LB...
1,1,376172,2025-06-05,Pacific Flood mporters INc,133.42,[{'description': 'YEAST INSTANT RED LBL SAF 20...
2,2,375991,2025-06-03,Pacific Flood Importers Inc,491.83,[{'description': 'FLOUR POWER 50 LB GRAINCRAFT...
3,3,20078834,2025-08-14,Frank's QQuality Produce,53.35,"[{'description': 'ONION, GREEN', 'quantity': 2..."
4,4,378483,2025-07-22,Pacific Flood mporters INc,576.63,"[{'description': 'FLOUR POWER GRAINCRAFT LB', ..."
5,5,20077622,2025-08-06,Frank's QQuality Produce,79.11,"[{'description': 'TOMATO, ROMA', 'quantity': 1..."
6,6,376739,2025-06-17,Pacific Food Importers Inc.,385.06,[{'description': 'FLOUR POWER GRAINCRAFT 50 LB...
7,7,375991,2025-06-03,Pacific Food Importers Inc.,491.83,[{'description': 'FLOUR POWER 50 LB GRAINCRAFT...
8,8,376355,2025-06-10,Pacific Food Importers Inc,688.51,"[{'description': 'FLOUR POWER GRAINCRAFT', 'qu..."
9,9,376172,2025-06-05,Pacific Food Importers Inc.,133.42,[{'description': 'YEAST INSTANT RED LBL SAF 20...


In [ ]:
df.to_csv("invoices.csv",index=False)

In [ ]:
import pandas as pd

In [ ]:
import json

with open("/content/all_invoices.json", "r") as file:
    data = json.load(file)

print(data)

[{'invoice_number': '378793', 'date': '07/29/2025', 'vendor': 'Pacific Food Importers Inc.', 'total_amount': 754.89, 'line_items': [{'description': 'FLOUR POWER GRAINCRAFT 50 LB', 'quantity': 10.0001, 'unit_price': 24.063, 'line_total': 240.0}, {'description': 'ONION MINCED MARCA CROC 4 * 4 .5 LB', 'quantity': 1.0, 'unit_price': 113.333, 'line_total': 113.33}, {'description': 'SESAME SEEDS MHITE+ MARCA CROC 4 LB', 'quantity': 1.0, 'unit_price': 80.25, 'line_total': 80.25}, {'description': 'SALT KOSHER COARSE DIAMOND CR 3 LB', 'quantity': 2.0, 'unit_price': 75.076, 'line_total': 150.15}, {'description': 'GLOVE NITRILE XL 10 100 ct', 'quantity': 1.0, 'unit_price': 36.25, 'line_total': 36.25}, {'description': 'YEAST INSTANT RED LBL SAF 20', 'quantity': 1.0, 'unit_price': 130.0, 'line_total': 130.0}]}, {'invoice_number': '376172', 'date': '06/05/2025', 'vendor': 'Pacific Flood mporters INc', 'total_amount': 133.42, 'line_items': [{'description': 'YEAST INSTANT RED LBL SAF 20.000 LB', 'quan

In [ ]:
import json
from typing import List, Dict


def load_json(file_path: str) -> List[Dict]:
    with open(file_path, "r") as f:
        data = json.load(f)
    return data



def extract_invoice_data(raw_data: List[Dict]) -> List[Dict]:

    structured_data = []

    for invoice in raw_data:

        invoice_info = {
            "invoice_number": invoice.get("invoice_number"),
            "date": invoice.get("date"),
            "vendor_name": invoice.get("vendor"),
            "total_amount": invoice.get("total_amount"),
            "line_items": []
        }

        # Extract line items
        for item in invoice.get("line_items", []):
            line_item = {
                "description": item.get("description"),
                "quantity": item.get("quantity"),
                "unit_price": item.get("unit_price"),
                "line_total": item.get("line_total")
            }
            invoice_info["line_items"].append(line_item)

        structured_data.append(invoice_info)

    return structured_data



def save_output(data: List[Dict], output_path: str):
    with open(output_path, "w") as f:
        json.dump(data, f, indent=4)



if __name__ == "__main__":

    input_file = "/content/all_invoices.json"
    output_file = "structured_invoices.json"

    raw_json = load_json(input_file)

    cleaned_data = extract_invoice_data(raw_json)

    save_output(cleaned_data, output_file)

    print("✅ Extraction completed. Output saved.")

✅ Extraction completed. Output saved.


In [ ]:
with open("/content/structured_invoices.json", "r") as file:
    data = json.load(file)


In [ ]:
data

[{'invoice_number': '378793',
  'date': '07/29/2025',
  'vendor_name': 'Pacific Food Importers Inc.',
  'total_amount': 754.89,
  'line_items': [{'description': 'FLOUR POWER GRAINCRAFT 50 LB',
    'quantity': 10.0001,
    'unit_price': 24.063,
    'line_total': 240.0},
   {'description': 'ONION MINCED MARCA CROC 4 * 4 .5 LB',
    'quantity': 1.0,
    'unit_price': 113.333,
    'line_total': 113.33},
   {'description': 'SESAME SEEDS MHITE+ MARCA CROC 4 LB',
    'quantity': 1.0,
    'unit_price': 80.25,
    'line_total': 80.25},
   {'description': 'SALT KOSHER COARSE DIAMOND CR 3 LB',
    'quantity': 2.0,
    'unit_price': 75.076,
    'line_total': 150.15},
   {'description': 'GLOVE NITRILE XL 10 100 ct',
    'quantity': 1.0,
    'unit_price': 36.25,
    'line_total': 36.25},
   {'description': 'YEAST INSTANT RED LBL SAF 20',
    'quantity': 1.0,
    'unit_price': 130.0,
    'line_total': 130.0}]},
 {'invoice_number': '376172',
  'date': '06/05/2025',
  'vendor_name': 'Pacific Flood mpor

In [ ]:
invoice_rows = []

for inv in data:
    invoice_rows.append({
        "invoice_number": inv.get("invoice_number"),
        "date": inv.get("date"),
        "vendor_name": inv.get("vendor_name"),
        "total_amount": inv.get("total_amount")
    })

invoice_df = pd.DataFrame(invoice_rows)



line_item_rows = []

for inv in data:
    invoice_number = inv.get("invoice_number")

    for item in inv.get("line_items", []):
        line_item_rows.append({
            "invoice_number": invoice_number,   # 🔗 foreign key
            "description": item.get("description"),
            "quantity": item.get("quantity"),
            "unit_price": item.get("unit_price"),
            "line_total": item.get("line_total")
        })

line_items_df = pd.DataFrame(line_item_rows)

In [ ]:
invoice_df

,invoice_number,date,vendor_name,total_amount
0,378793,07/29/2025,Pacific Food Importers Inc.,754.89
1,376172,06/05/2025,Pacific Flood mporters INc,133.42
2,375991,06/03/2025,Pacific Flood Importers Inc,491.83
3,20078834,8/14/2025,Frank's QQuality Produce,53.35
4,378483,07/22/2025,Pacific Flood mporters INc,576.63
5,20077622,8/6/2025,Frank's QQuality Produce,79.11
6,376739,06/17/2025,Pacific Food Importers Inc.,385.06
7,375991,06/03/2025,Pacific Food Importers Inc.,491.83
8,376355,06/10/2025,Pacific Food Importers Inc,688.51
9,376172,06/05/2025,Pacific Food Importers Inc.,133.42


In [ ]:
line_items_df

,invoice_number,description,quantity,unit_price,line_total
0,378793,FLOUR POWER GRAINCRAFT 50 LB,10.0001,24.063,240.00
1,378793,ONION MINCED MARCA CROC 4 * 4 .5 LB,1.0000,113.333,113.33
2,378793,SESAME SEEDS MHITE+ MARCA CROC 4 LB,1.0000,80.250,80.25
3,378793,SALT KOSHER COARSE DIAMOND CR 3 LB,2.0000,75.076,150.15
4,378793,GLOVE NITRILE XL 10 100 ct,1.0000,36.250,36.25
...,...,...,...,...,...
200,20071084,"CUCUMBER, XFANCY LG EA",12.0000,0.900,10.80
201,20071084,"ONION; ITALIAN RED # ONION, GREEN",6.0000,0.990,5.94
202,20071084,12BN BAG HERBS,2.0000,10.000,20.00
203,20071084,DILL BABY,1.0000,11.000,11.00


In [ ]:
invoice_df.to_csv("Invoices_df.csv",index=False)


In [ ]:
line_items_df.to_csv("line_items_df.csv",index=False)

In [ ]:
merged_invoices = pd.merge(line_items_df, invoice_df, on="invoice_number")

In [ ]:
merged_invoices

,invoice_number,description,quantity,unit_price,line_total,date,vendor_name,total_amount
0,378793,FLOUR POWER GRAINCRAFT 50 LB,10.0001,24.063,240.00,07/29/2025,Pacific Food Importers Inc.,754.89
1,378793,ONION MINCED MARCA CROC 4 * 4 .5 LB,1.0000,113.333,113.33,07/29/2025,Pacific Food Importers Inc.,754.89
2,378793,SESAME SEEDS MHITE+ MARCA CROC 4 LB,1.0000,80.250,80.25,07/29/2025,Pacific Food Importers Inc.,754.89
3,378793,SALT KOSHER COARSE DIAMOND CR 3 LB,2.0000,75.076,150.15,07/29/2025,Pacific Food Importers Inc.,754.89
4,378793,GLOVE NITRILE XL 10 100 ct,1.0000,36.250,36.25,07/29/2025,Pacific Food Importers Inc.,754.89
...,...,...,...,...,...,...,...,...
285,20071084,"CUCUMBER, XFANCY LG EA",12.0000,0.900,10.80,6/20/2025,Frank's QQuality Produce,79.12
286,20071084,"ONION; ITALIAN RED # ONION, GREEN",6.0000,0.990,5.94,6/20/2025,Frank's QQuality Produce,79.12
287,20071084,12BN BAG HERBS,2.0000,10.000,20.00,6/20/2025,Frank's QQuality Produce,79.12
288,20071084,DILL BABY,1.0000,11.000,11.00,6/20/2025,Frank's QQuality Produce,79.12


In [ ]:
merged_invoices.to_csv("full_Invoices.csv", index=False)